# Shallow Neural Networks & Universal Approximation Visualizer

Welcome to the interactive exploration of **Shallow Neural Networks**!

This notebook accompanies **Session 2 Notes** and brings the core mathematical principles to life through interactive charts:
1. **ReLU Activation Function** - Understanding non-linearity, pre-activations, and gradients.
2. **Single Layer Perceptron (SLP / Single Neuron)** - Exploring how $\theta_0, \theta_1, \phi_0, \phi_1$ shape the model output.
3. **Shallow Neural Network (MLP with 3 Neurons)** - Visualizing pre-activations $z_j$, activations $h_j = a[z_j]$, weighted contributions $\phi_j h_j$, and their final combination into a piecewise linear function.
4. **Universal Approximation Theorem** - An interactive linear regions visualizer demonstrating how adding hidden units allows a shallow network to approximate smooth continuous functions.


In [1]:
# Setup and Imports
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact, interactive, fixed, HBox, VBox, Layout, Label
from IPython.display import display, clear_output

# Configure high quality matplotlib style
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.size'] = 11
plt.rcParams['axes.edgecolor'] = '#cccccc'
plt.rcParams['axes.linewidth'] = 1.2
print("Environment successfully initialized!")


Environment successfully initialized!


---
## Section 1: Rectified Linear Unit (ReLU) Activation Function

The **Rectified Linear Unit (ReLU)** is defined as:

$$a[z] = \text{ReLU}[z] = \max(0, z) = \begin{cases} 0 & \text{if } z < 0 \\ z & \text{if } z \ge 0 \end{cases}$$

Where $z$ is the pre-activation $z = \theta_0 + \theta_1 x$. 
- For $z < 0$, the neuron is **inactive** (output is $0$, derivative $\frac{da}{dz} = 0$).
- For $z \ge 0$, the neuron is **active** (output passes $z$ linearly, derivative $\frac{da}{dz} = 1$).

Use the controls below to shift and scale the pre-activation input $z = \theta_1 x + \theta_0$.


In [2]:
def plot_relu_demo(theta0=0.0, theta1=1.0):
    plt.close('all')
    x = np.linspace(-5, 5, 500)
    z = theta1 * x + theta0
    a = np.maximum(0, z)
    da_dz = np.where(z > 0, 1.0, 0.0)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.2), dpi=100)
    
    # Left subplot: Activation function
    ax1.plot(x, z, '--', color='#888888', label='Pre-activation z(x) = theta0 + theta1 * x')
    ax1.plot(x, a, '-', color='#1f77b4', linewidth=2.5, label='Activated Output a[z] = ReLU[z]')
    ax1.axhline(0, color='black', lw=0.8, ls=':')
    ax1.axvline(0, color='black', lw=0.8, ls=':')
    
    # Calculate knot point x_knot where z = 0
    if theta1 != 0:
        x_knot = -theta0 / theta1
        if -5 <= x_knot <= 5:
            ax1.plot(x_knot, 0, 'ro', markersize=8, label=f'Knot Point (x = {x_knot:.2f})')
            ax1.axvline(x_knot, color='red', linestyle='--', alpha=0.5)
            
    ax1.set_title("Pre-activation z(x) vs ReLU Activation a[z]", fontsize=11, fontweight='bold')
    ax1.set_xlabel("Input x")
    ax1.set_ylabel("Value")
    ax1.set_ylim(-6, 6)
    ax1.legend(loc='upper left', fontsize=9)
    ax1.grid(True, alpha=0.3)
    
    # Right subplot: Gradient da/dz
    ax2.plot(x, da_dz, color='#e377c2', linewidth=2.5, label="Derivative da/dz")
    ax2.axhline(0, color='black', lw=0.8, ls=':')
    ax2.set_title("Gradient da/dz (Inactive vs Active Regions)", fontsize=11, fontweight='bold')
    ax2.set_xlabel("Input x")
    ax2.set_ylabel("Gradient da/dz")
    ax2.set_ylim(-0.2, 1.3)
    ax2.legend(loc='upper left', fontsize=9)
    ax2.grid(True, alpha=0.3)
    
    try:
        fig.tight_layout()
    except Exception:
        pass
    plt.show()

# Interactive controls
interact(
    plot_relu_demo,
    theta0=widgets.FloatSlider(value=0.0, min=-4.0, max=4.0, step=0.2, description='theta0 (Bias):'),
    theta1=widgets.FloatSlider(value=1.0, min=-3.0, max=3.0, step=0.2, description='theta1 (Weight):')
);


interactive(children=(FloatSlider(value=0.0, description='theta0 (Bias):', max=4.0, min=-4.0, step=0.2), Float…

---
## Section 2: Single Layer Perceptron (SLP / Single Neuron Model)

For a model with **1 hidden neuron**, the mathematical formulation is:

1. **Linear Pre-activation**:
   $$z = \theta_0 + \theta_1 x$$

2. **Hidden Neuron Activation**:
   $$h = a[z] = \text{ReLU}[\theta_0 + \theta_1 x]$$

3. **Model Output**:
   $$y = \phi_0 + \phi_1 h = \phi_0 + \phi_1 \text{ReLU}[\theta_0 + \theta_1 x]$$

A single ReLU neuron creates **1 inflection point (knot)**, dividing the 1D space into **2 distinct linear regions**:
- Region 1 (Inactive): $y = \phi_0$ (constant flat line with slope $0$).
- Region 2 (Active): $y = (\phi_0 + \phi_1 \theta_0) + (\phi_1 \theta_1) x$ (line with slope $\phi_1 \theta_1$).

Use the interactive sliders below to observe how $\theta_0, \theta_1$ position the knot and $\phi_0, \phi_1$ shift and scale the output.


In [3]:
def plot_slp_demo(theta0=-1.0, theta1=2.0, phi0=0.5, phi1=1.5):
    plt.close('all')
    x = np.linspace(-5, 5, 500)
    z = theta0 + theta1 * x
    h = np.maximum(0, z)
    y = phi0 + phi1 * h
    
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(14, 4.0), dpi=100)
    
    # Panel 1: Pre-activation z
    ax1.plot(x, z, color='#2ca02c', linewidth=2, label='z = theta0 + theta1 * x')
    ax1.axhline(0, color='black', lw=0.8, ls=':')
    ax1.set_title('1. Pre-activation z(x)', fontsize=11, fontweight='bold')
    ax1.set_xlabel('x')
    ax1.set_ylabel('z')
    ax1.grid(True, alpha=0.3)
    ax1.legend(loc='upper left', fontsize=9)
    
    # Panel 2: Hidden Activation h
    ax2.plot(x, h, color='#ff7f0e', linewidth=2.5, label='h = ReLU[z]')
    ax2.axhline(0, color='black', lw=0.8, ls=':')
    ax2.set_title('2. Activation h(x) = a[z]', fontsize=11, fontweight='bold')
    ax2.set_xlabel('x')
    ax2.set_ylabel('h')
    ax2.grid(True, alpha=0.3)
    ax2.legend(loc='upper left', fontsize=9)
    
    # Panel 3: Final Output y
    ax3.plot(x, y, color='#d62728', linewidth=2.8, label='y = phi0 + phi1 * h')
    ax3.axhline(0, color='black', lw=0.8, ls=':')
    
    # Highlight knot point and regions
    if theta1 != 0:
        x_knot = -theta0 / theta1
        if -5 <= x_knot <= 5:
            y_knot = phi0
            ax3.plot(x_knot, y_knot, 'ko', markersize=7, label=f'Knot Point (x={x_knot:.2f})')
            ax3.axvline(x_knot, color='gray', linestyle='--', alpha=0.6)
            
            # Region annotations
            ax3.text(x_knot - 2.2, y_knot + 0.5, "Region 1\n(Inactive)", fontsize=9, bbox=dict(boxstyle="round,pad=0.3", fc="#fff2cc", alpha=0.6))
            ax3.text(x_knot + 0.5, y_knot + 0.5, "Region 2\n(Active)", fontsize=9, bbox=dict(boxstyle="round,pad=0.3", fc="#e1f5fe", alpha=0.6))

    ax3.set_title('3. Output y = f[x, phi]', fontsize=11, fontweight='bold')
    ax3.set_xlabel('x')
    ax3.set_ylabel('y')
    ax3.grid(True, alpha=0.3)
    ax3.legend(loc='upper left', fontsize=9)
    
    try:
        fig.tight_layout()
    except Exception:
        pass
    plt.show()

interact(
    plot_slp_demo,
    theta0=widgets.FloatSlider(value=-1.0, min=-4.0, max=4.0, step=0.2, description='theta0:'),
    theta1=widgets.FloatSlider(value=2.0, min=-3.0, max=3.0, step=0.2, description='theta1:'),
    phi0=widgets.FloatSlider(value=0.5, min=-3.0, max=3.0, step=0.2, description='phi0:'),
    phi1=widgets.FloatSlider(value=1.5, min=-3.0, max=3.0, step=0.2, description='phi1:')
);


interactive(children=(FloatSlider(value=-1.0, description='theta0:', max=4.0, min=-4.0, step=0.2), FloatSlider…

---
## Section 3: Shallow Neural Network (MLP with 3 Hidden Neurons)

Expanding to **3 hidden neurons** in the single hidden layer:

$$\boldsymbol{\phi} = \{\phi_0, \phi_1, \phi_2, \phi_3, \theta_{10}, \theta_{11}, \theta_{20}, \theta_{21}, \theta_{30}, \theta_{31}\}$$

### Equations
1. **Pre-activations**:
   $$z_1 = \theta_{10} + \theta_{11} x, \quad z_2 = \theta_{20} + \theta_{21} x, \quad z_3 = \theta_{30} + \theta_{31} x$$

2. **Hidden Activations**:
   $$h_1 = a[z_1], \quad h_2 = a[z_2], \quad h_3 = a[z_3]$$

3. **Weighted Contributions**:
   $$c_1 = \phi_1 h_1, \quad c_2 = \phi_2 h_2, \quad c_3 = \phi_3 h_3$$

4. **Combined Output**:
   $$y = f[x, \boldsymbol{\phi}] = \phi_0 + \phi_1 h_1 + \phi_2 h_2 + \phi_3 h_3$$

With 3 ReLU units, there can be up to **3 knot points**, dividing the space into **up to 4 linear regions**!

Below is the **4-Panel Grid Visualizer** showing each phase step-by-step:
1. **Pre-activations ($z_1, z_2, z_3$)**
2. **Activations ($h_1, h_2, h_3 = a[z_j]$)**
3. **Weighted Components ($\phi_1 h_1, \phi_2 h_2, \phi_3 h_3$)**
4. **Final Combined Piecewise Linear Output ($y(x)$)**


In [4]:
def plot_mlp_demo(t10=-1.5, t11=1.5, t20=1.0, t21=-1.0, t30=-3.0, t31=1.0,
                  p0=0.0, p1=1.5, p2=1.2, p3=-1.8):
    plt.close('all')
    x = np.linspace(-5, 5, 500)
    
    # 1. Pre-activations
    z1 = t10 + t11 * x
    z2 = t20 + t21 * x
    z3 = t30 + t31 * x
    
    # 2. Hidden Activations
    h1 = np.maximum(0, z1)
    h2 = np.maximum(0, z2)
    h3 = np.maximum(0, z3)
    
    # 3. Weighted Components
    c1 = p1 * h1
    c2 = p2 * h2
    c3 = p3 * h3
    
    # 4. Final Output
    y = p0 + c1 + c2 + c3
    
    fig, axes = plt.subplots(2, 2, figsize=(13, 8.5), dpi=100)
    ax_z, ax_h, ax_c, ax_y = axes[0, 0], axes[0, 1], axes[1, 0], axes[1, 1]
    
    # Colors for neurons
    col1, col2, col3 = '#1f77b4', '#ff7f0e', '#2ca02c'
    
    # Panel 1: Pre-activations z_j
    ax_z.plot(x, z1, color=col1, ls='--', linewidth=1.8, label='z1 = theta10 + theta11 * x')
    ax_z.plot(x, z2, color=col2, ls='--', linewidth=1.8, label='z2 = theta20 + theta21 * x')
    ax_z.plot(x, z3, color=col3, ls='--', linewidth=1.8, label='z3 = theta30 + theta31 * x')
    ax_z.axhline(0, color='black', lw=0.8, ls=':')
    ax_z.set_title("1. Pre-activations (Linear Functions)", fontsize=11, fontweight='bold')
    ax_z.set_xlabel("x")
    ax_z.set_ylabel("z")
    ax_z.legend(loc='upper left', fontsize=9)
    ax_z.grid(True, alpha=0.3)
    
    # Panel 2: Activations h_j = a[z_j]
    ax_h.plot(x, h1, color=col1, linewidth=2.2, label='h1 = a[z1]')
    ax_h.plot(x, h2, color=col2, linewidth=2.2, label='h2 = a[z2]')
    ax_h.plot(x, h3, color=col3, linewidth=2.2, label='h3 = a[z3]')
    ax_h.axhline(0, color='black', lw=0.8, ls=':')
    ax_h.set_title("2. Hidden Activations hj = ReLU[zj]", fontsize=11, fontweight='bold')
    ax_h.set_xlabel("x")
    ax_h.set_ylabel("h")
    ax_h.legend(loc='upper left', fontsize=9)
    ax_h.grid(True, alpha=0.3)
    
    # Panel 3: Weighted Components phi_j * h_j
    ax_c.plot(x, c1, color=col1, linewidth=2.2, label='phi1 * h1')
    ax_c.plot(x, c2, color=col2, linewidth=2.2, label='phi2 * h2')
    ax_c.plot(x, c3, color=col3, linewidth=2.2, label='phi3 * h3')
    ax_c.axhline(0, color='black', lw=0.8, ls=':')
    ax_c.set_title("3. Weighted Components phi_j * h_j", fontsize=11, fontweight='bold')
    ax_c.set_xlabel("x")
    ax_c.set_ylabel("phi_j * h_j")
    ax_c.legend(loc='upper left', fontsize=9)
    ax_c.grid(True, alpha=0.3)
    
    # Panel 4: Combined Output y(x)
    ax_y.plot(x, y, color='#9467bd', linewidth=3.0, label='y = phi0 + sum(phi_j * h_j)')
    ax_y.axhline(0, color='black', lw=0.8, ls=':')
    
    # Calculate knot positions
    knots = []
    for t0, t1 in [(t10, t11), (t20, t21), (t30, t31)]:
        if t1 != 0:
            xk = -t0 / t1
            if -5 <= xk <= 5:
                knots.append(xk)
    
    knots.sort()
    for xk in knots:
        ax_y.axvline(xk, color='red', linestyle=':', alpha=0.7)
        
    ax_y.set_title(f"4. Combined Output (Up to {len(knots)+1} Linear Regions)", fontsize=11, fontweight='bold')
    ax_y.set_xlabel("x")
    ax_y.set_ylabel("y")
    ax_y.legend(loc='upper left', fontsize=9)
    ax_y.grid(True, alpha=0.3)
    
    try:
        fig.tight_layout()
    except Exception:
        pass
    plt.show()

# Create sliders for MLP parameters
style = {'description_width': '60px'}
layout = Layout(width='220px')

w_t10 = widgets.FloatSlider(value=-1.5, min=-4.0, max=4.0, step=0.5, description='theta10:', style=style, layout=layout)
w_t11 = widgets.FloatSlider(value=1.5, min=-3.0, max=3.0, step=0.5, description='theta11:', style=style, layout=layout)
w_t20 = widgets.FloatSlider(value=1.0, min=-4.0, max=4.0, step=0.5, description='theta20:', style=style, layout=layout)
w_t21 = widgets.FloatSlider(value=-1.0, min=-3.0, max=3.0, step=0.5, description='theta21:', style=style, layout=layout)
w_t30 = widgets.FloatSlider(value=-3.0, min=-4.0, max=4.0, step=0.5, description='theta30:', style=style, layout=layout)
w_t31 = widgets.FloatSlider(value=1.0, min=-3.0, max=3.0, step=0.5, description='theta31:', style=style, layout=layout)

w_p0 = widgets.FloatSlider(value=0.0, min=-3.0, max=3.0, step=0.5, description='phi0:', style=style, layout=layout)
w_p1 = widgets.FloatSlider(value=1.5, min=-3.0, max=3.0, step=0.5, description='phi1:', style=style, layout=layout)
w_p2 = widgets.FloatSlider(value=1.2, min=-3.0, max=3.0, step=0.5, description='phi2:', style=style, layout=layout)
w_p3 = widgets.FloatSlider(value=-1.8, min=-3.0, max=3.0, step=0.5, description='phi3:', style=style, layout=layout)

ui_neuron1 = VBox([Label(value="<b>Neuron 1 Params</b>"), w_t10, w_t11, w_p1])
ui_neuron2 = VBox([Label(value="<b>Neuron 2 Params</b>"), w_t20, w_t21, w_p2])
ui_neuron3 = VBox([Label(value="<b>Neuron 3 Params</b>"), w_t30, w_t31, w_p3])
ui_output = VBox([Label(value="<b>Output Bias</b>"), w_p0])

controls_ui = HBox([ui_neuron1, ui_neuron2, ui_neuron3, ui_output])

out = widgets.interactive_output(
    plot_mlp_demo,
    {'t10': w_t10, 't11': w_t11, 't20': w_t20, 't21': w_t21, 't30': w_t30, 't31': w_t31,
     'p0': w_p0, 'p1': w_p1, 'p2': w_p2, 'p3': w_p3}
)

display(controls_ui, out)


Output()

---
## Section 4: Universal Approximation Theorem (Linear Regions Visualizer)

### Theorem Statement
The **Universal Approximation Theorem** states that a shallow feedforward neural network with a single hidden layer containing a finite number of non-linear activation units (such as ReLU) can approximate any continuous function $f(x)$ on a compact interval to arbitrary accuracy $\epsilon > 0$.

### Key Intuition: Piecewise Linear Approximation
- Each ReLU hidden unit adds one **knot point** (inflection point).
- $N$ hidden units create **$N+1$ distinct linear regions**.
- Within each linear region, the network output is a straight line segment.
- **As $N$ (the number of linear regions slider) increases**, the length of each linear segment shrinks, and the piecewise linear curve gets **smoother and smoother**, rapidly converging to the exact target function!

Use the **Linear Regions Slider ($N$)** below to watch the piecewise approximation smooth out in real-time.


In [5]:
def target_func(x, choice="Sine Wave"):
    if choice == "Sine Wave":
        return np.sin(2 * x)
    elif choice == "Complex Oscillation":
        return np.sin(x) + 0.5 * np.cos(3 * x)
    elif choice == "Gaussian / Bell Curve":
        return 2 * np.exp(-x**2) - 1
    elif choice == "Non-linear Polynomial":
        return 0.1 * x**3 - 0.5 * x
    return np.sin(x)

def plot_uat_demo(num_regions=3, func_choice="Sine Wave"):
    plt.close('all')
    x = np.linspace(-3, 3, 1000)
    y_true = target_func(x, func_choice)
    
    # Construct N linear regions using piecewise linear interpolation (which is equivalent to exact N-neuron ReLU network output)
    x_knots = np.linspace(-3, 3, num_regions + 1)
    y_knots = target_func(x_knots, func_choice)
    y_approx = np.interp(x, x_knots, y_knots)
    
    # Calculate Mean Squared Error (MSE)
    mse = np.mean((y_true - y_approx)**2)
    
    fig, ax = plt.subplots(figsize=(12, 5.2), dpi=100)
    
    # Highlight individual linear regions with alternating background shading
    colors = ['#f0f8ff', '#fff5ee']
    for i in range(num_regions):
        ax.axvspan(x_knots[i], x_knots[i+1], color=colors[i % 2], alpha=0.5)
        ax.axvline(x_knots[i], color='gray', linestyle=':', alpha=0.6)
    ax.axvline(x_knots[-1], color='gray', linestyle=':', alpha=0.6)
    
    # Plot true continuous curve vs ReLU network piecewise approximation
    ax.plot(x, y_true, color='#1f77b4', linewidth=3.0, label=f'Target Function: {func_choice}')
    ax.plot(x, y_approx, color='#d62728', linewidth=2.2, linestyle='-', label=f'Shallow Network ({num_regions} Linear Regions)')
    ax.plot(x_knots, y_knots, 'ko', markersize=5, label='Knot Points')
    
    ax.set_title(f"Universal Approximation: {num_regions} Linear Regions | MSE = {mse:.6f}", 
                 fontsize=12, fontweight='bold', pad=12)
    ax.set_xlabel("Input Domain x", fontsize=11)
    ax.set_ylabel("Output y", fontsize=11)
    ax.set_xlim(-3.1, 3.1)
    ax.legend(loc='upper right', frameon=True, facecolor='white', framealpha=0.9, fontsize=10)
    ax.grid(True, alpha=0.3)
    
    try:
        fig.tight_layout()
    except Exception:
        pass
    plt.show()

interact(
    plot_uat_demo,
    num_regions=widgets.IntSlider(value=3, min=1, max=50, step=1, description='Linear Regions (N):', style={'description_width': '140px'}, layout=Layout(width='500px')),
    func_choice=widgets.Dropdown(
        options=["Sine Wave", "Complex Oscillation", "Gaussian / Bell Curve", "Non-linear Polynomial"],
        value="Sine Wave",
        description="Target Function:",
        style={'description_width': '140px'},
        layout=Layout(width='350px')
    )
);


interactive(children=(IntSlider(value=3, description='Linear Regions (N):', layout=Layout(width='500px'), max=…